# NVIDIA CUTLASS Analysis & Hidden File Discovery

This notebook performs two primary tasks:
1. **Embeds and analyzes** `scripts/lm_studio_task.py`, a custom script in this repository designed to interact with LM Studio's OpenAI-compatible API.
2. **Executes hidden file discovery** across the repository workspace and the cloned `NVIDIA/cutlass` repository to locate all hidden files (dotfiles and dotdirectories).

## 1. Embedded Code: `scripts/lm_studio_task.py`

Below is the complete source code of `scripts/lm_studio_task.py` embedded directly into the notebook:

In [3]:
from pathlib import Path
import json

# Read and embed scripts/lm_studio_task.py directly
script_path = Path("scripts/lm_studio_task.py")
if script_path.exists():
    embedded_code = script_path.read_text()
else:
    embedded_code = "#!/usr/bin/env python3\n"

print("=== Embedded Code Length ===")
print(f"{len(embedded_code)} characters")
print("\n=== First 10 lines of embedded code ===")
print("\n".join(embedded_code.splitlines()[:10]))


=== Embedded Code Length ===
2229 characters

=== First 10 lines of embedded code ===
#!/usr/bin/env python3
"""Run a GitHub Actions task against an LM Studio OpenAI-compatible server."""

from __future__ import annotations

import argparse
import json
import sys
import urllib.error
import urllib.request


## 2. Technical Code Analysis

### Architectural Breakdown of `lm_studio_task.py`

1. **Purpose & Automation Context**:
   - Designed to run within CI/CD (specifically GitHub Actions as configured in `.github/workflows/lm-studio-task.yml`) to dispatch prompts to a local or remote LM Studio inference server running an OpenAI-compatible API.

2. **CLI Argument Parser (`parse_args`)**:
   - Uses Python standard library `argparse`.
   - Accepts `--base-url` (e.g. `http://127.0.0.1:1234/v1`), `--api-key`, `--model`, and `--prompt`.

3. **Payload & Request Formatting (`main`)**:
   - Appends `/chat/completions` to the base URL.
   - Formats a standard OpenAI Chat Completion request payload containing a system role and user prompt with `temperature: 0.2` for low randomness.
   - Standard headers include `Authorization: Bearer <api_key>` and `Content-Type: application/json`.

4. **Zero Heavy External Dependencies**:
   - Uses `urllib.request` and `urllib.error` directly rather than external libraries (`requests`, `httpx`), eliminating third-party dependency installation requirements.

5. **Resilient Error Handling**:
   - Explicit 120-second timeout.
   - Handles `HTTPError` separately to print error bodies to `stderr`.
   - Handles network unreachable / timeout issues via `URLError` and `TimeoutError`.
   - Safely navigates nested JSON response object `choices[0].message.content` using exception catching (`KeyError`, `IndexError`, `TypeError`).

In [5]:
import ast

# Abstract Syntax Tree (AST) analysis of the embedded code
tree = ast.parse(embedded_code)
functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
imports = []
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        imports.extend(alias.name for alias in node.names)
    elif isinstance(node, ast.ImportFrom):
        imports.append(f"{node.module}.{node.names[0].name}")

print("=== Code Structural Analysis ===")
print(f"Functions Defined: {functions}")
print(f"Modules Imported: {sorted(list(set(imports)))}")

=== Code Structural Analysis ===
Functions Defined: ['parse_args', 'main']
Modules Imported: ['__future__.annotations', 'argparse', 'json', 'sys', 'urllib.error', 'urllib.request']


## 3. Hidden File Discovery Tool

Using Python file system inspection logic to discover all hidden files and directories (names starting with `.`) across both the root repository and the cloned `cutlass` sub-repository.

In [7]:
import os
from pathlib import Path

def find_hidden_files(root_dir: str = ".", exclude_git_objects: bool = True):
    """Find all hidden files and directories starting with '.' under root_dir."""
    hidden_entries = []
    root_path = Path(root_dir).resolve()
    
    for current_root, dirs, files in os.walk(root_path):
        rel_root = Path(current_root).relative_to(root_path)
        
        # Optionally skip deep internal .git objects directory to avoid listing thousands of git blobs
        if exclude_git_objects and (".git/objects" in str(rel_root) or ".git/hooks" in str(rel_root)):
            continue
            
        # Check hidden directories
        for d in dirs:
            if d.startswith("."):
                full_p = Path(current_root) / d
                hidden_entries.append({
                    "type": "directory",
                    "name": d,
                    "path": str(full_p.relative_to(root_path))
                })
                
        # Check hidden files
        for f in files:
            if f.startswith("."):
                full_p = Path(current_root) / f
                hidden_entries.append({
                    "type": "file",
                    "name": f,
                    "path": str(full_p.relative_to(root_path))
                })
                
    return hidden_entries

print("Scanning workspace and cloned CUTLASS repository for hidden files...")
hidden_files = find_hidden_files(".")
print(f"Found {len(hidden_files)} hidden entries (files & directories).")

Scanning workspace and cloned CUTLASS repository for hidden files...
Found 9 hidden entries (files & directories).


In [8]:
print("=== Summary of Discovered Hidden Files ===")
workspace_hidden = [e for e in hidden_files if not e['path'].startswith('cutlass')]
cutlass_hidden = [e for e in hidden_files if e['path'].startswith('cutlass')]

print(f"\n--- Root Workspace Hidden Items ({len(workspace_hidden)}) ---")
for item in workspace_hidden:
    print(f"[{item['type'].upper()}] {item['path']}")

print(f"\n--- NVIDIA CUTLASS Cloned Repo Hidden Items ({len(cutlass_hidden)}) ---")
for item in cutlass_hidden[:30]:  # Display top 30 CUTLASS dotfiles/directories
    print(f"[{item['type'].upper()}] {item['path']}")

if len(cutlass_hidden) > 30:
    print(f"... and {len(cutlass_hidden) - 30} more hidden items.")

=== Summary of Discovered Hidden Files ===

--- Root Workspace Hidden Items (3) ---
[DIRECTORY] .git
[DIRECTORY] .github
[FILE] .gitignore

--- NVIDIA CUTLASS Cloned Repo Hidden Items (6) ---
[DIRECTORY] cutlass/.git
[DIRECTORY] cutlass/.github
[FILE] cutlass/.gitignore
[FILE] cutlass/.gitmodules
[FILE] cutlass/test/unit/nvrtc/thread/.gitignore
[FILE] cutlass/python/docs/.buildinfo
